# Informed Align-then-Unlearn — Qwen3.5-4B on Colab Pro (A100 80GB)

End-to-end pipeline: causal tracing to identify the most responsible decoder layer for a target, then align-then-unlearn at that layer with layer-scoped gradient unfreezing.

**Runtime:** Colab Pro → *Runtime → Change runtime type → A100 GPU, High-RAM*.

**Pipeline (all interactive prompts are in cells 2 and 3 — after those you can walk away):**
1. Check GPU
2. Mount Drive (interactive — auth popup)
3. Install + login W&B (interactive — paste API key)
4. Clone repo + install deps (upgrades `transformers` for Qwen3.5)
5. Download RWKU benchmark
6. Retokenize `positive_*.json` for the Qwen tokenizer
7. Causal tracing for `1_Stephen_King`
8. Read the peak layer
9. Smoke-test training (1 epoch)
10. Full chained align-then-unlearn

## 1. Verify the GPU

You want to see `A100-SXM4-80GB` (or `A100 80GB PCIe`). If you see a T4 or V100, switch the runtime.

In [ ]:
!nvidia-smi

## 2. Mount Drive and pin `HF_HOME`  ⚠️ interactive

Auth popup. Persists Qwen3.5-4B weights (~8GB) across sessions so re-runs don't re-download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

## 3. Install + login W&B  ⚠️ interactive

Metrics get logged to the `informed-align-unlearn` project. This is the **last** interactive cell — after you paste your key, everything downstream runs unattended.

Grab your key from https://wandb.ai/authorize. If you don't want W&B, comment out `wandb.login()` and add `wandb.mode=disabled` to the training overrides in cells 18/20.

In [ ]:
!pip install -q wandb
import wandb
wandb.login()

## 4. Clone the repo (branch `shiv`) and install dependencies

The `shiv` branch has: causal tracing, Qwen3.5-4B config, layer-scoped unfreezing, chained training, tokenizer-variant data loading.

We also upgrade `transformers` to the latest — `requirements.txt` pins `4.51.3` which doesn't know about Qwen3.5's `qwen3_5` model type.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d informed-align-unlearn ]; then
  git clone https://github.com/AyushSehgal/informed-align-unlearn.git
fi
cd informed-align-unlearn
git fetch origin shiv
git checkout shiv
git pull origin shiv
pip install -q -r requirements.txt
pip install -q --upgrade transformers

## 5. Download the RWKU benchmark

Pulls the benchmark zip from Google Drive (~a few hundred MB) and unpacks it into `data/rwku/benchmark/`.

In [ ]:
%%bash
cd /content/informed-align-unlearn
bash data/rwku/download_rwku_data.sh
echo '---'
ls data/rwku/benchmark/Target | head

## 6. Retokenize `positive_phi.json` for the Qwen tokenizer

The repo ships with `positive_phi.json` (raw text). The Qwen loader expects `positive_qwen.json`, so we copy-and-filter by the Qwen tokenizer's length budget. 1024 tokens is a safe cap for the ATU training loop.

In [ ]:
%%bash
cd /content/informed-align-unlearn
python data/rwku/retokenize_positive.py \
  --tokenizer Qwen/Qwen3.5-4B \
  --variant qwen \
  --max_tokens 1024 \
  --trust_remote_code

## 7. Causal tracing for `1_Stephen_King`

ROME-style clean / corrupted / restored forward passes, averaged over RWKU `forget_level{1,2,3}.json` prompts. Writes `outputs/causal_trace/1_Stephen_King/results.json`.

Takes ~10–15 min on an A100 for Qwen3.5-4B. To change the target, edit `--target_id` (e.g. `9_Justin_Bieber`).

In [ ]:
%%bash
cd /content/informed-align-unlearn
bash scripts/run_causal_trace.sh \
  --target_id 1_Stephen_King \
  --model Qwen/Qwen3.5-4B \
  --top_k 5 \
  --noise_multiplier 3.0 \
  --local

## 8. Read the peak layer out of `results.json`

The training step uses this layer as the unlearning hook. The Python variable `peak` gets interpolated into the next shell command via Colab's `{var}` syntax.

In [ ]:
import json
from pathlib import Path

results = json.loads(
    Path('/content/informed-align-unlearn/outputs/causal_trace/1_Stephen_King/results.json').read_text()
)
peak = int(results['peak_layer'])
print('peak_layer =', peak)
print('top_k_layers:')
for entry in results['top_k_layers']:
    print(f"  layer {entry['layer']:>3d}  recovery {entry['recovery']:.4f}")

## 9. Smoke test — 1 epoch, skip the pre-eval

Quick sanity check that the full pipeline loads cleanly (model, data, layer hook, trainer) before committing to a long run. Should finish in a couple of minutes on an A100.

In [ ]:
!cd /content/informed-align-unlearn && bash scripts/run_chained_training.sh --chain "1_Stephen_King:{peak}" --overrides "trainer.max_epochs=1 skip_initial_eval=true" --local

## 10. Full chained align-then-unlearn

Single target, single layer — starting point. Once this works, you can extend the chain:

```
--chain "1_Stephen_King:4,2_Confucius:20,3_Elon_Musk:10"
```

Each step's modified model is fed into the next. Eval metrics are prefixed `pre_chain/` and `post_chain/` in the W&B dashboard so you can see USR/GUR/APR move per target.

In [ ]:
!cd /content/informed-align-unlearn && bash scripts/run_chained_training.sh --chain "1_Stephen_King:{peak}" --local

## Colab gotchas

- **Session disconnects after ~90 min of idle.** Keep the tab open. Colab Pro+ has background execution; regular Pro does not.
- **A100 vs. L4/V100.** If you end up on a smaller GPU, force `torch_dtype: "float16"` in `config/pre_trained_llm/qwen3.5-4b.yaml` or switch to a smaller model.
- **`flash_attn` import errors.** Qwen3.5-4B is loaded with `attn_implementation="eager"` to avoid flash-attn kernel compile issues on Colab. Don't change without installing flash-attn wheels.
- **Drive I/O is slow.** `HF_HOME` on Drive is fine for one-time downloads, but Lightning checkpoints stay on `/content/` to avoid stalls. Default `save_dir` already does this.
- **Disk fill.** Colab Pro gives ~200GB scratch — plenty for one model + a few checkpoints. Set `task.training_module.checkpoint_interval=-1` to skip intermediate checkpoints if you chain many targets.
- **`trust_remote_code`.** The Qwen config sets this to `true` and the causal trace script auto-adds the flag. Don't override unless you know what you're doing.
- **RWKU download flakes.** `gdown` occasionally throws a quota error. Re-run the cell — it's idempotent.